# Repeatable Data Quality Framework for Pricing Intelligence

This notebook treats the Online Retail II dataset as a simulated first customer data handover for Pryzm Solutions.

Goal:
- assess data quality
- identify pricing model blockers
- define repeatable intake checks
- create business-focused recommendations for revenue, cost, and customer service impact

## 1. Load and Combine Source Data

Purpose:
- load both yearly Excel sheets
- combine them into one transaction dataset
- create one repeatable basis for all quality checks

In [1]:
import pandas as pd

file_path = "/content/online_retail_II.xlsx"

df_2009 = pd.read_excel(file_path, sheet_name="Year 2009-2010")
df_2010 = pd.read_excel(file_path, sheet_name="Year 2010-2011")

df = pd.concat([df_2009, df_2010], ignore_index=True)

print("Rows and columns:", df.shape)
df.head()

Rows and columns: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Data Structure and Completeness Check

Purpose:
- check rows, columns and data types
- identify missing values
- flag missing customer or product information
- assess whether the data is usable for pricing analysis

In [3]:
# Basic dataset overview
print("Rows and columns:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

# Missing values summary
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().sum() / len(df) * 100).round(2)
}).sort_values(by="missing_percentage", ascending=False)

missing_summary

Rows and columns: (1067371, 8)

Column names:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Data types:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object


,missing_count,missing_percentage
Customer ID,243007,22.77
Description,4382,0.41
StockCode,0,0.00
Invoice,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Country,0,0.00


## 3. Transaction Status and Value Validation

Purpose:
- separate sales, cancellations and returns
- identify invalid quantities or prices
- avoid distorted revenue, demand and pricing signals

In [4]:
# Create transaction quality flags
df["is_cancelled_invoice"] = df["Invoice"].astype(str).str.startswith("C")
df["is_negative_quantity"] = df["Quantity"] < 0
df["is_zero_or_negative_price"] = df["Price"] <= 0
df["line_revenue"] = df["Quantity"] * df["Price"]

transaction_quality_summary = pd.DataFrame({
    "check": [
        "Cancelled invoices",
        "Negative quantities",
        "Zero or negative prices",
        "Rows with negative revenue"
    ],
    "row_count": [
        df["is_cancelled_invoice"].sum(),
        df["is_negative_quantity"].sum(),
        df["is_zero_or_negative_price"].sum(),
        (df["line_revenue"] < 0).sum()
    ]
})

transaction_quality_summary["percentage"] = (
    transaction_quality_summary["row_count"] / len(df) * 100
).round(2)

transaction_quality_summary

,check,row_count,percentage
0,Cancelled invoices,19494,1.83
1,Negative quantities,22950,2.15
2,Zero or negative prices,6207,0.58
3,Rows with negative revenue,19498,1.83


## 4. Duplicate Detection

Purpose:
- identify repeated transaction rows
- avoid overstated demand, revenue and customer activity
- reduce manual cleaning effort during onboarding

In [5]:
# Duplicate row check
duplicate_count = df.duplicated().sum()
duplicate_percentage = round(duplicate_count / len(df) * 100, 2)

duplicate_summary = pd.DataFrame({
    "check": ["Duplicate rows"],
    "row_count": [duplicate_count],
    "percentage": [duplicate_percentage]
})

duplicate_summary

,check,row_count,percentage
0,Duplicate rows,34335,3.22


## 5. Product Consistency Check

Purpose:
- check whether product codes map consistently to descriptions
- identify weak product master data
- reduce risk of wrong product-level pricing logic

In [6]:
# Product consistency: how many descriptions per StockCode?
product_description_counts = (
    df.dropna(subset=["Description"])
      .groupby("StockCode")["Description"]
      .nunique()
      .reset_index(name="unique_descriptions")
)

inconsistent_products = product_description_counts[
    product_description_counts["unique_descriptions"] > 1
]

product_consistency_summary = pd.DataFrame({
    "check": [
        "StockCodes with multiple descriptions",
        "Maximum descriptions for one StockCode"
    ],
    "value": [
        len(inconsistent_products),
        product_description_counts["unique_descriptions"].max()
    ]
})

product_consistency_summary

,check,value
0,StockCodes with multiple descriptions,1232
1,Maximum descriptions for one StockCode,9


## 6. Outlier Check

Purpose:
- identify unusually high or low quantities and prices
- reduce risk of distorted revenue and demand signals
- flag records requiring validation before pricing analysis

In [7]:
# Outlier check using simple percentile thresholds
quantity_limits = df["Quantity"].quantile([0.01, 0.99])
price_limits = df["Price"].quantile([0.01, 0.99])

quantity_outliers = df[
    (df["Quantity"] < quantity_limits.loc[0.01]) |
    (df["Quantity"] > quantity_limits.loc[0.99])
]

price_outliers = df[
    (df["Price"] < price_limits.loc[0.01]) |
    (df["Price"] > price_limits.loc[0.99])
]

outlier_summary = pd.DataFrame({
    "check": [
        "Quantity below 1st percentile",
        "Quantity above 99th percentile",
        "Price below 1st percentile",
        "Price above 99th percentile"
    ],
    "threshold": [
        quantity_limits.loc[0.01],
        quantity_limits.loc[0.99],
        price_limits.loc[0.01],
        price_limits.loc[0.99]
    ],
    "row_count": [
        (df["Quantity"] < quantity_limits.loc[0.01]).sum(),
        (df["Quantity"] > quantity_limits.loc[0.99]).sum(),
        (df["Price"] < price_limits.loc[0.01]).sum(),
        (df["Price"] > price_limits.loc[0.99]).sum()
    ]
})

outlier_summary["percentage"] = (
    outlier_summary["row_count"] / len(df) * 100
).round(2)

outlier_summary

,check,threshold,row_count,percentage
0,Quantity below 1st percentile,-3.00,9668,0.91
1,Quantity above 99th percentile,100.00,10561,0.99
2,Price below 1st percentile,0.21,10599,0.99
3,Price above 99th percentile,18.00,10614,0.99


## 7. Commercial Risk Scorecard

Purpose:
- translate technical data issues into business risk
- show impact on revenue, cost, customer service and model readiness
- create a repeatable output for future customer onboarding

In [8]:
risk_scorecard = pd.DataFrame({
    "risk_area": [
        "Revenue readiness",
        "Revenue readiness",
        "Cost-to-clean risk",
        "Customer service risk",
        "Model readiness",
        "Model readiness"
    ],
    "data_issue": [
        "Missing Customer ID",
        "Missing cost, margin and discount context",
        "Duplicate rows",
        "Inconsistent product descriptions",
        "Cancellations and negative quantities",
        "Quantity and price outliers"
    ],
    "finding": [
        "22.77% of rows missing Customer ID",
        "Not available in source data",
        "3.22% duplicate rows",
        "1,232 StockCodes have multiple descriptions",
        "1.83% cancelled invoices; 2.15% negative quantities",
        "Approx. 1% high/low tails for quantity and price"
    ],
    "business_impact": [
        "Blocks customer-level pricing and segmentation",
        "Blocks profitability-based pricing and price leakage analysis",
        "Can overstate revenue, demand and customer activity",
        "Weakens product-level pricing and increases manual cleanup",
        "Distorts demand, revenue and model signals if mixed with sales",
        "Can distort pricing recommendations if not validated"
    ],
    "priority": [
        "High",
        "Critical",
        "Medium",
        "Medium",
        "High",
        "Medium"
    ]
})

risk_scorecard

,risk_area,data_issue,finding,business_impact,priority
0,Revenue readiness,Missing Customer ID,22.77% of rows missing Customer ID,Blocks customer-level pricing and segmentation,High
1,Revenue readiness,"Missing cost, margin and discount context",Not available in source data,Blocks profitability-based pricing and price l...,Critical
2,Cost-to-clean risk,Duplicate rows,3.22% duplicate rows,"Can overstate revenue, demand and customer act...",Medium
3,Customer service risk,Inconsistent product descriptions,"1,232 StockCodes have multiple descriptions",Weakens product-level pricing and increases ma...,Medium
4,Model readiness,Cancellations and negative quantities,1.83% cancelled invoices; 2.15% negative quant...,"Distorts demand, revenue and model signals if ...",High
5,Model readiness,Quantity and price outliers,Approx. 1% high/low tails for quantity and price,Can distort pricing recommendations if not val...,Medium


## 8. Customer Intake Specification

Purpose:
- define the minimum data Pryzm should request from future customers
- reduce repeated manual cleaning
- make onboarding more scalable and model-ready

In [9]:
intake_specification = pd.DataFrame({
    "field": [
        "Customer_ID",
        "Customer_Name",
        "Customer_Segment",
        "SKU_Product_ID",
        "Product_Description",
        "Product_Category",
        "Invoice_ID",
        "Invoice_Date",
        "Quantity",
        "Unit_Price",
        "Currency",
        "Cost",
        "Margin",
        "Discount",
        "Contract_Price",
        "Return_Cancellation_Flag",
        "Country_Region",
        "Sales_Channel",
        "Source_System",
        "Data_Extraction_Date"
    ],
    "required": [
        "Yes", "Recommended", "Recommended", "Yes", "Yes", "Recommended",
        "Yes", "Yes", "Yes", "Yes", "Yes", "Critical", "Critical",
        "Recommended", "Recommended", "Yes", "Yes", "Recommended",
        "Yes", "Yes"
    ],
    "reason": [
        "Customer-level pricing and segmentation",
        "Customer validation and communication",
        "Segment-based pricing logic",
        "Product-level pricing and matching",
        "Human-readable product validation",
        "Product hierarchy and pricing groups",
        "Transaction traceability",
        "Time-based pricing and trend analysis",
        "Demand and volume analysis",
        "Pricing analysis",
        "Cross-market price comparison",
        "Profitability-based pricing",
        "Margin protection",
        "Price leakage and discount analysis",
        "Contract pricing validation",
        "Separate sales from returns/cancellations",
        "Market and regional pricing analysis",
        "Channel-specific pricing analysis",
        "Traceability to ERP/CRM/source export",
        "Repeatable onboarding and audit trail"
    ],
    "example_validation_check": [
        "Not null; stable ID",
        "Not null if available",
        "Valid segment list",
        "Not null; consistent format",
        "Not null; consistent with SKU",
        "Valid category list",
        "Not null; unique with line number",
        "Valid date format",
        "Numeric; positive for sales",
        "Numeric; greater than 0 for sales",
        "ISO currency code",
        "Numeric; greater than or equal to 0",
        "Numeric; valid margin range",
        "Numeric; valid discount range",
        "Numeric; matches contract terms",
        "Boolean or defined transaction status",
        "Valid country or region",
        "Valid channel list",
        "ERP, CRM or other system name",
        "Valid date"
    ]
})

intake_specification

,field,required,reason,example_validation_check
0,Customer_ID,Yes,Customer-level pricing and segmentation,Not null; stable ID
1,Customer_Name,Recommended,Customer validation and communication,Not null if available
2,Customer_Segment,Recommended,Segment-based pricing logic,Valid segment list
3,SKU_Product_ID,Yes,Product-level pricing and matching,Not null; consistent format
4,Product_Description,Yes,Human-readable product validation,Not null; consistent with SKU
5,Product_Category,Recommended,Product hierarchy and pricing groups,Valid category list
6,Invoice_ID,Yes,Transaction traceability,Not null; unique with line number
7,Invoice_Date,Yes,Time-based pricing and trend analysis,Valid date format
8,Quantity,Yes,Demand and volume analysis,Numeric; positive for sales
9,Unit_Price,Yes,Pricing analysis,Numeric; greater than 0 for sales


## 9. Repeatable Onboarding Runbook

Purpose:
- turn one-off onboarding into a repeatable process
- show what Pryzm should do before model training
- reduce manual work and customer clarification loops

In [10]:
onboarding_runbook = pd.DataFrame({
    "step": [
        1, 2, 3, 4, 5, 6, 7, 8
    ],
    "action": [
        "Receive customer export",
        "Confirm source systems and extraction date",
        "Run automated data quality checks",
        "Score commercial and model-readiness risks",
        "Send customer feedback on missing or invalid fields",
        "Receive corrected or enriched data",
        "Approve dataset for pricing model preparation",
        "Document repeated issues for product improvement"
    ],
    "output": [
        "Raw customer handover",
        "Source-system overview",
        "Quality summary",
        "Risk scorecard",
        "Customer feedback list",
        "Improved dataset",
        "Model-ready dataset",
        "Reusable onboarding learnings"
    ]
})

onboarding_runbook

,step,action,output
0,1,Receive customer export,Raw customer handover
1,2,Confirm source systems and extraction date,Source-system overview
2,3,Run automated data quality checks,Quality summary
3,4,Score commercial and model-readiness risks,Risk scorecard
4,5,Send customer feedback on missing or invalid f...,Customer feedback list
5,6,Receive corrected or enriched data,Improved dataset
6,7,Approve dataset for pricing model preparation,Model-ready dataset
7,8,Document repeated issues for product improvement,Reusable onboarding learnings


## 10. Recommendation

Purpose:
- define the first improvement Pryzm should prioritize
- connect data quality to revenue, cost and customer service impact

In [11]:
recommendation = pd.DataFrame({
    "priority": ["First improvement"],
    "recommendation": [
        "Build an automated Customer Data Quality Scorecard before onboarding starts"
    ],
    "why_it_matters": [
        "It reduces manual cleaning, speeds up onboarding, improves customer feedback and protects pricing model quality"
    ],
    "commercial_impact": [
        "Higher revenue readiness, lower cost-to-clean, better customer service and safer model training"
    ]
})

recommendation

,priority,recommendation,why_it_matters,commercial_impact
0,First improvement,Build an automated Customer Data Quality Score...,"It reduces manual cleaning, speeds up onboardi...","Higher revenue readiness, lower cost-to-clean,..."


## 11. Final Summary

Key findings:
- 22.77% of rows are missing Customer ID, limiting customer-level pricing and segmentation.
- 1.83% of rows are cancelled invoices and 2.15% contain negative quantities, which must be separated from normal sales.
- 3.22% of rows are duplicates, creating risk of overstated demand, revenue and customer activity.
- 1,232 StockCodes have multiple descriptions, indicating weak product master data.
- Cost, margin, discount and contract fields are missing, which blocks profitability-based pricing.

Business impact:
- Revenue risk: pricing decisions cannot reliably identify price leakage, margin opportunities or customer-level value.
- Cost risk: repeated manual cleaning increases onboarding effort.
- Customer service risk: missing or unclear data creates clarification loops with customers.
- Model risk: pricing models may learn from distorted or incomplete transaction data.

Recommended first improvement:
- Build a reusable Customer Data Quality Scorecard before onboarding starts.
- Use it to classify each customer handover by revenue readiness, cost-to-clean risk, customer service risk and model readiness.
- Send customers a clear feedback list before model preparation begins.

## 12. Margin Leakage Readiness

Key point:
- The dataset can show transaction activity, but it cannot fully support margin-improvement decisions yet.

What is missing:
- product cost
- gross margin or target margin
- list price vs. net price
- discount information
- contract price
- customer segment
- product category

Business impact:
- Revenue can be calculated, but profitability cannot be assessed.
- Low prices cannot be classified as discounts, contract terms, promotions or data errors.
- A pricing model trained on this data alone could learn sales behavior, but not true margin potential.
- For Pryzm’s use case, cost, margin and discount context should be mandatory before model training.

## 13. Example Customer Feedback

Based on this first handover, Pryzm could send the customer the following feedback:

- Customer IDs are missing in 22.77% of rows. Stable customer identifiers are required for customer-level pricing and segmentation.
- Cost, margin and discount fields are missing. These fields are required to identify margin leakage and pricing improvement potential.
- Cancellations and returns should be provided as a separate transaction status field, not mixed with normal sales.
- Product master data should be cleaned so that each SKU has one stable product description and ideally one product category.
- Duplicate rows should be reviewed before pricing analysis because they can overstate demand, revenue and customer activity.

Purpose:
- reduce repeated clarification loops
- speed up onboarding
- improve trust in the pricing recommendations
- make the next customer export more model-ready